# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane 4 — CTR / Engagement Opportunity Scoring.**

The question this lane asks is: *"Which visible pages under-capture clicks or engagement and deserve a review?"* That is a **"which ones first?"** question, not a yes/no question — so per the framing table it maps to **ranking / scoring**, not classification.

Why not classification: a binary label ("opportunity / not") would have to come from a rule I define myself (e.g. a CTR-gap threshold), and the framing skill is explicit that a target defined by my own rule just teaches the model my rule, not the world. A continuous **opportunity score** (how far a page sits below what its position tier normally earns) avoids inventing a fake ground truth and still produces the ranked list an editor needs.

Why not clustering: clustering answers "what kinds of pages exist," which is Lane 3's question. Here I already know the unit (a page) and the axis I care about (CTR relative to its peers) — I don't need to discover unlabeled groups first.

**The four framing questions:**
1. **Decision improved:** which page an editor should open and fix (title/meta/snippet/on-page) *first*, this week.
2. **Who acts, and how:** a content editor or SEO reviewer works down the ranked queue, top to bottom, and opens the flagged pages.
3. **Cost of a wrong answer:** a false positive wastes an editor's review time on a page that's actually fine; a false negative leaves real click-loss unreviewed for another cycle. Neither is catastrophic — so this is a *prioritization* tool, not a high-stakes decision.
4. **Why ML/data over a fixed rule:** expected CTR is not one number — it shifts by position tier, content type, and intent, and those factors interact. A single hard-coded threshold can't track that; see Section 5 for the evidence.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
print("rows, cols:", df.shape)
print("clients:", df["client_id"].nunique())


rows, cols: (30000, 44)
clients: 32


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (proxy): `ctr_gap`** — the difference between a page's own observed `ctr` and the **median observed CTR of its `position_tier`, computed only over pages with real traffic volume** (its peer group, noise-filtered).

```
ctr_gap = ctr - median(ctr | position_tier, impressions_90d >= 500)
```

The volume filter on the *baseline* matters, not just on the candidate page: with all pages included, low-traffic pages with a handful of impressions and zero clicks drag the tier median all the way down to 0.00 for `top_3` and `deep`, which would make `ctr_gap` meaningless for those tiers (checked this empirically below — it's a real artifact, not a hypothetical). Restricting the *median calculation* to pages with >=500 impressions produces a sensible, monotonic gradient across tiers (page_1 > top_3 > striking > page_3_5 > deep) that actually reflects "what a well-trafficked page in this tier normally earns."

A large negative `ctr_gap` on a page with real volume (enough impressions) means it earns fewer clicks than peers sitting in the *same* rank range — that's the opportunity signal.

**Is this observed or defined?** It's a proxy, and I want to be honest about that distinction rather than paper over it: `ctr` itself is a fully **observed** outcome (Google Search Console clicks/impressions over 90 days — nothing about it is invented). What I *define* is the comparison group (`position_tier`) used to compute the gap. That's a much lighter, more defensible kind of "definition" than the rejected alternative — a hand-picked pass/fail threshold turning CTR into a 0/1 label. Grouping by an already-observed, already-existing column (`position_tier`) and taking a gap against real peer medians keeps the target continuous and tied to real numbers, rather than manufacturing a label the model would just learn to reproduce.

This also directly avoids the lane's #1 listed mistake: *"comparing CTR across different positions without adjusting for position."*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
valid = df[df["avg_position"] > 0].copy()

# First, show WHY an unfiltered median is broken: low-volume noise pages drag it to 0
unfiltered_median = valid.groupby("position_tier")["ctr"].median()
print("Median CTR by tier, NO volume filter (broken -- see top_3/deep):")
print(unfiltered_median.sort_values(ascending=False))
print()

# Fix: compute the tier baseline only from pages with real traffic volume
MIN_IMPRESSIONS_FOR_BASELINE = 500
trafficked = valid[valid["impressions_90d"] >= MIN_IMPRESSIONS_FOR_BASELINE]
tier_median_ctr = trafficked.groupby("position_tier")["ctr"].median()
print(f"Median CTR by tier, volume-filtered (>= {MIN_IMPRESSIONS_FOR_BASELINE} impressions) -- the real baseline:")
print(tier_median_ctr.sort_values(ascending=False))

valid["ctr_gap"] = valid["ctr"] - valid["position_tier"].map(tier_median_ctr)
valid[["content_id", "position_tier", "impressions_90d", "ctr", "ctr_gap"]].sort_values("ctr_gap").head(10)



Median CTR by tier, NO volume filter (broken -- see top_3/deep):
position_tier
page_1      0.16
striking    0.11
page_3_5    0.03
deep        0.00
top_3       0.00
Name: ctr, dtype: float64

Median CTR by tier, volume-filtered (>= 500 impressions) -- the real baseline:
position_tier
page_1      0.24
top_3       0.20
striking    0.17
page_3_5    0.09
deep        0.00
Name: ctr, dtype: float64


,content_id,position_tier,impressions_90d,ctr,ctr_gap
2877,content_1fa9912d0c02,page_1,63,0.0,-0.24
26963,content_445e559e0148,page_1,22,0.0,-0.24
2850,content_19c88f333c6f,page_1,6,0.0,-0.24
9768,content_d94334a6b69f,page_1,9,0.0,-0.24
9765,content_70a28e3eaa24,page_1,220,0.0,-0.24
9651,content_3c62618cdd85,page_1,28,0.0,-0.24
9647,content_e978bc7251c0,page_1,42,0.0,-0.24
2882,content_5d64fc00babd,page_1,470,0.0,-0.24
9684,content_45709db72fb3,page_1,65,0.0,-0.24
20055,content_a7d58e84079e,page_1,498,0.0,-0.24


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@K** (matching the lane guide's suggested metric and the starter pipeline's own reference metric, `Precision@50`).

Concretely: take the top-K pages by opportunity score (most negative `ctr_gap`, filtered to pages with enough impression volume to not be noise), and check what fraction of those K pages, on manual/downstream review, turn out to be genuine review-worthy cases (high impressions, position stable enough that low CTR isn't just noise, and gap large enough to matter).

Why this metric and not something like RMSE or overall accuracy: an editor only ever works through a fixed number of pages per week (their queue capacity, K). What matters isn't how well-calibrated the score is everywhere — it's whether the *top of the queue* is worth their time. Precision@K measures exactly that action-relevant slice, ignoring the tail of pages nobody will ever get to. The starter pipeline's own baseline-vs-model comparison (Precision@50 ≈ 0.24 → 0.74) is a direct example of this metric being used to justify moving from a rule to a model.

"Good" here means Precision@K meaningfully beats a naive baseline (e.g. "just sort by raw CTR ascending" or "just sort by impressions descending") — not an absolute number in isolation.

**Honest caveat, checked in code below:** the position-adjusted top-50 queue turns out to be 100% `page_1` pages. That's not a bug — `page_1` simply has far more volume-qualified pages (7,064) than any other tier (`top_3` has only 458), so it's statistically expected to produce more extreme outliers just from having more draws. A real version of this scoring approach would need within-tier standardization (e.g. a z-score, not a raw gap) so tiers with more rows don't structurally dominate the queue — flagging that as a next iteration, not hiding it.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
K = 50
min_impressions = 500  # filter out low-volume noise per lane guide's "common mistake"

candidates = valid[valid["impressions_90d"] >= min_impressions].copy()

naive_queue = candidates.sort_values("ctr").head(K)
gap_queue = candidates.sort_values("ctr_gap").head(K)

print(f"Naive (raw CTR) top-{K} -- position_tier mix:")
print(naive_queue["position_tier"].value_counts())
print()
print(f"Position-adjusted (ctr_gap) top-{K} -- position_tier mix:")
print(gap_queue["position_tier"].value_counts())



Naive (raw CTR) top-50 -- position_tier mix:
position_tier
page_3_5    20
striking    12
page_1      10
deep         5
top_3        3
Name: count, dtype: int64

Position-adjusted (ctr_gap) top-50 -- position_tier mix:
position_tier
page_1    50
Name: count, dtype: int64


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
**One row = one content page** (`content_id`), pseudonymized, belonging to one client (`client_id`), with its trailing-90-day search performance. This matches the starter dataset's stated grain directly (30,000 rows Ã— 44 columns, one row per pseudonymized content item).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane4_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "ctr", "avg_position", "position_tier",
    "impression_tier", "engagement_rate", "scroll_rate", "sessions_90d",
]
lane4_slice = valid[lane4_cols].copy()

print("Unit of analysis: one row = one content page")
print("Shape:", lane4_slice.shape)
lane4_slice.head(10)



Unit of analysis: one row = one content page
Shape: (28795, 13)


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,ctr,avg_position,position_tier,impression_tier,engagement_rate,scroll_rate,sessions_90d
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,0.76,10.6,striking,good,5.88,4.55,17
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,0.05,20.3,page_3_5,good,0.00,10.00,9
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,0.09,36.5,page_3_5,good,0.00,28.57,11
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,58,0.49,6.2,page_1,good,1.28,3.45,78
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,24,0.13,44.0,page_3_5,good,0.00,24.29,145
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3970,1,0.03,8.5,page_1,good,0.00,25.00,5
6,content_9a34b442b552,client_8722616204,keyword article,informational,20,0,0.00,7.0,page_1,low,0.00,0.00,1
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,1724,1,0.06,21.2,page_3_5,moderate,3.57,7.14,28
8,content_5e6c160719bc,client_6208ef0f77,keyword article,informational,32574,29,0.09,46.0,page_3_5,excellent,5.88,6.25,68
9,content_c27558df2b0c,client_19581e27de,keyword article,informational,1240,2,0.16,4.9,page_1,moderate,0.00,0.00,3


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
A fixed rule like `flag if ctr < 0.3` sounds simple, but the data itself shows why it breaks: "normal" CTR is not one number, it moves a lot depending on where a page ranks. In this slice, median CTR for `top_3` is roughly 5x higher than `page_3_5`, and `deep` sits lower still. A single global threshold either over-flags every page stuck deep in the results (structurally low CTR, nothing wrong with the page) or under-flags high-position pages quietly leaking clicks (their "low" CTR is still numerically higher than the global cutoff, so the rule never catches them).

The code below makes this concrete: the same global CTR threshold produces wildly different flag rates depending only on which position tier a page happens to sit in — proof that one rule can't serve every tier fairly, and that the comparison has to be tier-aware (which is exactly what `ctr_gap` does, and what a fixed if-statement doesn't).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
GLOBAL_THRESHOLD = valid["ctr"].median()  # one "reasonable-looking" global cutoff
print(f"Global CTR median used as a naive fixed threshold: {GLOBAL_THRESHOLD:.2f}")

flag_rate_by_tier = (
    valid.assign(flagged=valid["ctr"] < GLOBAL_THRESHOLD)
    .groupby("position_tier")["flagged"]
    .mean()
    .sort_values(ascending=False)
)
print("\nShare of pages flagged by ONE global rule, split by position tier:")
print((flag_rate_by_tier * 100).round(1).astype(str) + "%")
print("\n-> A single if-statement flags almost every 'deep'/'page_3_5' page (structural, not a real")
print("   problem) while barely touching 'top_3' pages that may be leaking real clicks -- ")
print("   evidence the pattern needs a tier-aware comparison, not one hard-coded number.")

Global CTR median used as a naive fixed threshold: 0.08

Share of pages flagged by ONE global rule, split by position tier:
position_tier
deep        87.3%
page_3_5    59.5%
top_3       57.6%
striking    45.2%
page_1      39.1%
Name: flagged, dtype: object

-> A single if-statement flags almost every 'deep'/'page_3_5' page (structural, not a real
   problem) while barely touching 'top_3' pages that may be leaking real clicks -- 
   evidence the pattern needs a tier-aware comparison, not one hard-coded number.


## Self-check

Before you submit, confirm each line honestly:


- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.